# ARTI406: Assignment 2 - Advanced Data Preprocessing & Unsupervised PCA Transformations
**Student Name:** Sara Mohammad Othman  
**Student ID:** 2230009061

**Course Code:** ARTI406  
**Dataset Variant:** Mall Customers Segmentation Matrix

## Task 1: Identify Data Quality Issues
In this task, we execute structural and programmatic diagnostics on the `Mall_Customers.csv` dataset to check for any inconsistencies, data type mismatches, missing values, or boundary violations.

In [8]:
!pip install scikit-learn

In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA


df = pd.read_csv('Mall_Customers.csv')

print('=== Task 1: Data Quality Structural Diagnostics ===')
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Duplicate Row Count: {df.duplicated().sum()} duplicated rows found.\n')

print('=== Column Character/Data Type Constraints ===')
print(df.dtypes)

print('\n=== Numerical Value Boundary & Range Auditing ===')
print(df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].describe().loc[['min', 'max']])

print('\n=== Missing Values Checklist ===')
print(df.isnull().sum())

=== Task 1: Data Quality Structural Diagnostics ===
Dataset Shape: 200 rows, 5 columns
Duplicate Row Count: 0 duplicated rows found.

=== Column Character/Data Type Constraints ===
CustomerID                int64
Gender                      str
Age                       int64
Annual Income (k$)        int64
Spending Score (1-100)    int64
dtype: object

=== Numerical Value Boundary & Range Auditing ===
      Age  Annual Income (k$)  Spending Score (1-100)
min  18.0                15.0                     1.0
max  70.0               137.0                    99.0

=== Missing Values Checklist ===
CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64


## Task 2: Apply One Missing Value Strategy and Explain Why
### Strategy Selection: Median Imputation
**Justification:** The empirical dataset naturally contains 0 missing values. To demonstrate pipeline engineering capabilities, a synthetic missing value (NaN) is programmatically injected into the `Annual Income (k$)` column. 

We choose **Median Imputation** over Mean Imputation because the median is a robust statistical metric that is highly resilient against the skewness and outliers often present in income distributions.

In [14]:
# ---------------------------------------------------------------------
# Task 2: Apply One Missing Value Strategy and Explain Why
# ---------------------------------------------------------------------
print('=== Task 2: Missing Value Strategy (Median Imputation) ===')

# 1. Inject a synthetic missing value to demonstrate pipeline execution capabilities
df.loc[15, 'Annual Income (k$)'] = np.nan
print(f'Value at Index 15 after NaN injection: {df.loc[15, "Annual Income (k$)"]}')

# 2. Apply Median Imputation pipeline strategy
median_income = df['Annual Income (k$)'].median()
df['Annual Income (k$)'] = df['Annual Income (k$)'].fillna(median_income)

print('\n=== Imputation Process Matrix ===')
print(f'Calculated Imputation Metric (Median): {median_income}')
print(f'Repaired Dataset State at Row Index 15: {df.loc[15, "Annual Income (k$)"]}')
print(f'Current Global Null Values Remaining: {df.isnull().sum().sum()}')

=== Task 2: Missing Value Strategy (Median Imputation) ===
Value at Index 15 after NaN injection: nan

=== Imputation Process Matrix ===
Calculated Imputation Metric (Median): 62.0
Repaired Dataset State at Row Index 15: 62.0
Current Global Null Values Remaining: 0


## Task 3: Detect and Handle Outliers Using IQR
We utilize the Interquartile Range (IQR) method to establish statistical boundaries. Any data point falling beyond $Q3 + 1.5 \times IQR$ or $Q1 - 1.5 \times IQR$ is flagged as an outlier. 

To handle these outliers without losing structural patterns, we apply **Statistical Capping (Winsorization)**, which clips values directly at the calculated boundary thresholds.

In [19]:
# Calculate statistical bounds using the Interquartile Range framework
Q1 = df['Annual Income (k$)'].quantile(0.25)
Q3 = df['Annual Income (k$)'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Annual Income (k$)'] < lower_bound) | (df['Annual Income (k$)'] > upper_bound)]
print('=== Statistical Boundary Configuration (IQR Auditing) ===')
print(f'IQR: {IQR} | Lower Bound: {lower_bound} | Upper Bound: {upper_bound}')
print(f'\nOutlier Matrix Identified:\n{outliers}')

# Handle Outliers using statistical capping (Winsorization)
df['Annual Income (k$)'] = np.clip(df['Annual Income (k$)'], lower_bound, upper_bound)
print(f'\nPost-Capping Maximum Check value: {df["Annual Income (k$)"].max()}')

=== Statistical Boundary Configuration (IQR Auditing) ===
IQR: 36.0 | Lower Bound: -12.0 | Upper Bound: 132.0

Outlier Matrix Identified:
Empty DataFrame
Columns: [CustomerID, Gender, Age, Annual Income (k$), Spending Score (1-100)]
Index: []

Post-Capping Maximum Check value: 132.0


## Task 4: Normalize Numerical Features Using Both Min-Max and Z-score
Data transformation normalization scales raw features to ensure unified numeric ranges. 
* **Min-Max Scaling:** Bounds numbers precisely between $[0, 1]$.
* **Z-Score Standardization:** Shifts values to have a mean ($\mu = 0$) and standard deviation ($\sigma = 1$).

In [20]:
# ---------------------------------------------------------------------
# Task 4: Normalize Numerical Features Using Both Min-Max and Z-score
# ---------------------------------------------------------------------
print('=== Task 4: Feature Normalization ===')
numerical_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

# 1. Apply MinMax Scaling (Transforms values to a range between 0 and 1)
minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(minmax_scaler.fit_transform(df[numerical_cols]), columns=[c+'_minmax' for c in numerical_cols])

# 2. Apply Standard Z-score Scaling (Transforms values to have Mean=0 and SD=1)
z_scaler = StandardScaler()
df_zscore = pd.DataFrame(z_scaler.fit_transform(df[numerical_cols]), columns=[c+'_zscore' for c in numerical_cols])

# 3. Concatenate vectors for preview verification
df_normalized = pd.concat([df['CustomerID'], df_minmax, df_zscore], axis=1)
print('Scaled Processing Array Output Sample (Head):')
df_normalized.head()

=== Task 4: Feature Normalization ===
Scaled Processing Array Output Sample (Head):


,CustomerID,Age_minmax,Annual Income (k$)_minmax,Spending Score (1-100)_minmax,Age_zscore,Annual Income (k$)_zscore,Spending Score (1-100)_zscore
0,1,0.019231,0.000000,0.387755,-1.424569,-1.765328,-0.434801
1,2,0.057692,0.000000,0.816327,-1.281035,-1.765328,1.195704
2,3,0.038462,0.008547,0.051020,-1.352802,-1.726716,-1.715913
3,4,0.096154,0.008547,0.775510,-1.137502,-1.726716,1.040418
4,5,0.250000,0.017094,0.397959,-0.563369,-1.688104,-0.395980


## Task 5: Apply PCA and Interpret Explained Variance
Principal Component Analysis (PCA) reduces dataset dimensionality while maintaining maximum statistical variance. We apply PCA to the Z-score standardized features (`df_zscore`) to project our 3-dimensional numeric space (`Age`, `Annual Income`, `Spending Score`) onto 2 core principal components.

In [21]:
# ---------------------------------------------------------------------
# Task 5: Apply PCA and Interpret Explained Variance
# ---------------------------------------------------------------------
print('=== Task 5: Principal Component Analysis (PCA) ===')

# 1. Initialize PCA to project data into 2 Principal Components
pca = PCA(n_components=2)
pca_transformed = pca.fit_transform(df_zscore)

# 2. Output the variance explained by each component
print(f'Explained Variance Ratio Per Component: {pca.explained_variance_ratio_}')
print(f'Cumulative Model Retained Explained Variance: {np.sum(pca.explained_variance_ratio_):.4f}')

# 3. Output Principal Component Structural Weight Matrix (Loadings)
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=numerical_cols)
print('\n=== Principal Component Structural Weight Matrix (Loadings) ===')
print(loadings)

=== Task 5: Principal Component Analysis (PCA) ===
Explained Variance Ratio Per Component: [0.4432308  0.33251396]
Cumulative Model Retained Explained Variance: 0.7757

=== Principal Component Structural Weight Matrix (Loadings) ===
                             PC1       PC2
Age                     0.704622  0.057549
Annual Income (k$)     -0.086156  0.996270
Spending Score (1-100) -0.704333 -0.064294
